# 00 — Python, NumPy, Pandas & ML Primer (Loan Risk Prediction)

Welcome to the **Loan Risk Prediction** project!  
This notebook is a self‑contained, beginner‑friendly tutorial that takes you from near‑zero Python experience to training and evaluating a baseline machine learning model on tabular loan data.

**You will learn:**
- Jupyter basics and Python essentials (types, lists/dicts, loops, functions, modules)
- NumPy arrays, shapes, vectorization, broadcasting
- Pandas for tabular data (load, inspect, clean, transform)
- Plotting with Matplotlib
- A complete tabular ML workflow:
  - Load raw data
  - Clean/transform features (including categorical encoding)
  - Train/validation split with stratification
  - Scale numeric features
  - Train **Logistic Regression**
  - Evaluate with **accuracy, precision, recall, F1, ROC‑AUC**, confusion matrix, ROC curve
  - (Optional) Cross‑validation and calibration

> If you can run this notebook top‑to‑bottom, you have everything you need for this project.

## Prerequisites & Environment Check

- **Python 3.13.5** (recommended for this repo)
- A virtual environment activated for this project (e.g., `python -m venv .venv` then `source .venv/bin/activate` on macOS/Linux or `.venv\Scripts\activate` on Windows)
- Required packages installed: `pip install -r requirements.txt`

> If your Python is slightly different (e.g., 3.12), this tutorial still works, but we standardize on 3.13.5 for consistency.

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## Using This Notebook

- **Run a cell**: Click it and press `Shift+Enter` (or the ▶️ run button).
- **Restart the kernel** if imports behave oddly: Kernel → Restart Kernel.
- **Tip**: Keep outputs small and readable; print only what helps you learn.

## 1. Python Essentials (15–25 min)

This section covers just enough Python to be productive in the rest of the project.

### 1.1 Variables and Basic Types
Python has dynamic typing. Common types you'll use right away:
- `int`, `float`, `bool`, `str`
- Containers: `list`, `tuple`, `dict`, `set`

In [ ]:
# ints, floats, bools, strings
year = 2025
pi = 3.14159
is_student = True
greeting = "hello, mdst!"

print(year, type(year))
print(pi, type(pi))
print(is_student, type(is_student))
print(greeting, type(greeting))

# Lists (mutable)
nums = [3, 1, 4, 1, 5]
nums.append(9)
print("nums:", nums)

# Tuples (immutable)
pt = (10, 20)
print("pt:", pt)

# Dicts (key-value maps)
person = {"name": "Raymond", "role": "Project Lead", "year": 2}
print("person:", person)

# Sets (unique elements)
uniq = set([1,2,2,3])
print("uniq:", uniq)

### 1.2 Control Flow and Loops

In [ ]:
# if / elif / else
score = 87
if score >= 90:
    letter = 'A'
elif score >= 80:
    letter = 'B'
else:
    letter = 'C'
print("letter:", letter)

# for loop
total = 0
for x in [1, 2, 3, 4]:
    total += x
print("total:", total)

# while loop
n = 5
fact = 1
while n > 0:
    fact *= n
    n -= 1
print("5!:", fact)

### 1.3 Functions & Comprehensions

In [ ]:
def greet(name: str = "world") -> str:
    return f"hello, {name}!"

print(greet("mdst"))

# List comprehension
squares = [x*x for x in range(6)]
print("squares:", squares)

# Dict comprehension
square_map = {x: x*x for x in range(6)}
print("square_map:", square_map)

### 1.4 Exceptions (brief)

In [ ]:
try:
    x = 1 / 0
except ZeroDivisionError as e:
    print("Caught:", e)

## 2. NumPy Basics (10–20 min)

**NumPy** provides fast multi‑dimensional arrays and vectorized math. It is the foundation for scientific Python.

In [ ]:
import numpy as np

# Create arrays
a = np.array([1, 2, 3])
b = np.array([[1., 2., 3.],
              [4., 5., 6.]])
print("a:", a, "shape:", a.shape)
print("b:\n", b, "shape:", b.shape)

# Dtypes and casting
print("b dtype:", b.dtype)
b_int = b.astype(np.int64)
print("b_int dtype:", b_int.dtype)

# Vectorized operations
print("a + 10:", a + 10)
print("b * 2:", b * 2)
print("dot:", a @ a)  # dot product

### Indexing, Slicing, Broadcasting

In [ ]:
x = np.arange(12).reshape(3,4)
print("x:\n", x)

# Slicing rows/cols
print("x[0, :]:", x[0, :])
print("x[:, 2]:", x[:, 2])

# Broadcasting (add a column vector to each row)
col = np.array([[100],[200],[300]])
print("x + col:\n", x + col)

### Random & Reproducibility

In [ ]:
rng = np.random.default_rng(seed=42)
randn = rng.normal(loc=0.0, scale=1.0, size=(2,3))
print(randn)

## 3. Pandas for Tabular Data (20–30 min)

**Pandas** is your go‑to for CSVs, data cleaning, and feature engineering.

In [ ]:
import pandas as pd

# Build a tiny demo DataFrame
df_demo = pd.DataFrame({
    "Loan ID": [1,2,3,4],
    "Customer ID": [111, 222, 333, 444],
    "Current Loan Amount": [5000, 100_000_000, 12000, 7000],
    "Term": ["Short", "Long", "Long", "Short"],
    "Home Ownership": ["Rent", "Mortgage", "Own", "Rent"],
    "Annual Income": [50000, np.nan, 82000, 60000],
    "Loan Status": ["Fully Paid", "Defaulted", "Current", "Charged Off"]
})
df_demo

In [ ]:
# Inspect structure
print(df_demo.info())
display(df_demo.describe(numeric_only=True))
display(df_demo.head())

### Common Cleaning Tasks
- Drop unhelpful IDs
- Fix **sentinel** values (e.g., bogus 100,000,000)
- Handle missing values
- Map target labels to 0/1

In [ ]:
# 1) Drop IDs (they don't help the model predict outcome)
df = df_demo.drop(columns=["Loan ID", "Customer ID"], errors="ignore").copy()

# 2) Fix sentinel for 'Current Loan Amount' (if present)
if "Current Loan Amount" in df.columns:
    df.loc[df["Current Loan Amount"] == 100_000_000, "Current Loan Amount"] = np.nan

# 3) Map target labels
TARGET_MAP = {
    "Defaulted": 1, "Charged Off": 1, "Charged-off": 1, "Delinquent": 1,
    "Fully Paid": 0, "Paid off": 0, "Current": 0, "Approved": 0
}
df["Loan Status"] = df["Loan Status"].map(TARGET_MAP)

display(df)
print("Class balance (counts):\n", df["Loan Status"].value_counts(dropna=False))

In [ ]:
# 4) Handle missing values (simple example)
# Numeric: fill with median; Categorical: fill with 'Missing'
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols and c != "Loan Status"]

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for c in cat_cols:
    df[c] = df[c].astype("string").fillna("Missing")

display(df.head())

### Quick Visual Checks
(We'll use **Matplotlib** directly—it's what you'll use for ROC curves later.)

In [ ]:
from matplotlib import pyplot as plt

# Histogram of a numeric feature
col = "Annual Income"
if col in df.columns:
    plt.figure()
    df[col].hist(bins=15)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col); plt.ylabel("Count")
    plt.show()

## 4. From DataFrame → Features

We'll separate the **target** (`y`) from the **features** (`X`), then encode categoricals and scale numerics.

In [ ]:
# Separate target
y = df["Loan Status"].astype(int).values
X = df.drop(columns=["Loan Status"], errors="ignore")

print("X shape:", X.shape, "| y shape:", y.shape)
print("X dtypes:\n", X.dtypes)

In [ ]:
# Column groupings
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

num_cols, cat_cols

In [ ]:
# Build preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

numeric_transformer = RobustScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)
preprocess

## 5. Train/Validation Split (Stratified)

We use a **stratified** split so the class ratio (defaults vs non‑defaults) is similar in both train and validation sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape, " Valid size:", X_valid.shape)
print("Class balance train:", {0: (y_train==0).sum(), 1: (y_train==1).sum()})
print("Class balance valid:", {0: (y_valid==0).sum(), 1: (y_valid==1).sum()})

## 6. Train a Baseline Model: Logistic Regression

We'll wrap the preprocessor and the estimator in a single **Pipeline**.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=None))
])

pipe

In [ ]:
pipe.fit(X_train, y_train)
print("Train score (accuracy):", pipe.score(X_train, y_train))
print("Valid score (accuracy):", pipe.score(X_valid, y_valid))

## 7. Evaluation: Metrics & Curves

We care about **probabilities** for ROC‑AUC and thresholds, so we’ll use `predict_proba` for the positive class (1 = bad outcome).  
**Important:** `y_prob = model.predict_proba(X)[:, 1]` is used for ROC and PR curves.

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

# Class predictions & probabilities
y_pred = pipe.predict(X_valid)
y_prob = pipe.predict_proba(X_valid)[:, 1]

print("Confusion matrix:\n", confusion_matrix(y_valid, y_pred))
print("\nClassification report:\n", classification_report(y_valid, y_pred))
print("ROC-AUC (using y_prob):", roc_auc_score(y_valid, y_prob))

# ROC curve
RocCurveDisplay.from_predictions(y_valid, y_prob)
plt.title("Validation ROC Curve")
plt.show()

# Precision-Recall curve
PrecisionRecallDisplay.from_predictions(y_valid, y_prob)
plt.title("Validation Precision-Recall Curve")
plt.show()

### Interpreting the Metrics
- **Accuracy**: overall correctness; can be misleading if classes are imbalanced.
- **Precision (for class 1)**: out of predicted defaults, how many were actually defaults?
- **Recall (for class 1)**: out of actual defaults, how many did we catch?
- **F1**: harmonic mean of precision and recall.
- **ROC‑AUC**: threshold‑independent measure of ranking quality; **use `y_prob`**.

## 8. (Optional) Cross‑Validation

Use **StratifiedKFold** to get a more stable estimate than a single split.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
print("CV ROC-AUC scores:", auc_scores)
print("Mean ROC-AUC:", auc_scores.mean())

## 9. (Optional) Probability Calibration

If you need well‑calibrated probabilities (e.g., a predicted 0.2 should mean ~20% default rate), wrap the model with **CalibratedClassifierCV**.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone

# Fit base pipeline, then calibrate on validation split
base = clone(pipe)
base.fit(X_train, y_train)

calibrated = CalibratedClassifierCV(base, method="isotonic", cv="prefit")
calibrated.fit(X_valid, y_valid)

y_prob_cal = calibrated.predict_proba(X_valid)[:, 1]
print("Calibrated ROC-AUC:", roc_auc_score(y_valid, y_prob_cal))

RocCurveDisplay.from_predictions(y_valid, y_prob_cal)
plt.title("Validation ROC Curve (Calibrated)")
plt.show()

## 10. Saving Processed Data & Model (for the rest of the repo)

Often we save intermediate artifacts so training notebooks can load them quickly.

In [ ]:
from pathlib import Path

out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

# Save train/valid splits as CSV (after simple cleaning but before one-hot/scale)
X_train_out = X_train.copy()
X_train_out["Loan Status"] = y_train
X_valid_out = X_valid.copy()
X_valid_out["Loan Status"] = y_valid

X_train_out.to_csv(out_dir / "train_split.csv", index=False)
X_valid_out.to_csv(out_dir / "valid_split.csv", index=False)

print("Wrote:", (out_dir / "train_split.csv").resolve())
print("Wrote:", (out_dir / "valid_split.csv").resolve())

> Later notebooks in this repo (e.g., `final_preprocessing_data.ipynb`, `regression_model.ipynb`) can read these files, or you can repeat similar steps there with the full dataset.

## 11. Checkpoints & Mini‑Exercises

1. **Python Warm‑Up**
   - Write a function `only_even(nums)` that returns a new list containing only the even numbers from `nums`.
2. **NumPy**
   - Create a `(100, )` array of random numbers and compute its mean and standard deviation.
3. **Pandas**
   - Using `df_demo`, compute the average `Annual Income` grouped by `Home Ownership`.
4. **Modeling**
   - Change the `LogisticRegression` to `LogisticRegression(C=0.5)` and compare ROC‑AUC.
5. **Visualization**
   - Make a histogram of `Current Loan Amount` and a boxplot of `Annual Income`.

In [ ]:
# 1) only_even
def only_even(nums):
    return [x for x in nums if x % 2 == 0]

print(only_even([1,2,3,4,5,6]))

# 2) NumPy stats
rng = np.random.default_rng(0)
arr = rng.normal(size=100)
print("mean:", arr.mean(), "std:", arr.std())

# 3) Pandas groupby
if "Home Ownership" in df_demo.columns:
    display(df_demo.groupby("Home Ownership")["Annual Income"].mean())

## 12. Next Steps

- Replace the tiny `df_demo` with your real dataset (e.g., `credit_train.csv` from the repo).
- Reuse the **preprocessing**, **split**, and **pipeline** patterns shown here.
- Move to the project notebooks:
  - `final_preprocessing_data.ipynb` – more complete cleanup/feature engineering
  - `regression_model.ipynb` – train, evaluate, and compare models

> If you get stuck, post in **Slack (Loan Risk Prediction F25)** with a screenshot of the error and the cell you ran.